# Delta Lake Time Travel and Rollback

This notebook demonstrates Delta Lake's time travel capabilities. We'll:

1. Query the table history using DESCRIBE HISTORY
2. Intentionally write bad data (negative sales, corrupted records)
3. Query previous versions using VERSION AS OF and TIMESTAMP AS OF
4. Compare data across versions
5. Rollback to a previous version using RESTORE TABLE
6. Verify the rollback was successful
7. Demonstrate the ability to query across time for analytical purposes
8. Show how to optimize storage while preserving time travel capabilities

## 1. Initialize Spark Session with Delta Lake

In [ ]:
import os
import sys
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr, lit, current_timestamp
from delta.tables import DeltaTable
from datetime import datetime, timedelta

# Add scripts directory to path
sys.path.append('/opt/spark/scripts')
import utils

# Create Spark session with Delta Lake support
spark = SparkSession.builder \
    .appName("Delta Lake Time Travel") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

## 2. Load Delta Table and Get Initial Metrics

In [ ]:
# Define Delta table path
delta_table_path = "/opt/spark/data/processed/global_superstore_delta"

# Load the Delta table
delta_table = DeltaTable.forPath(spark, delta_table_path)

# Get initial table metrics
initial_metrics = utils.log_delta_table_metrics(spark, delta_table_path)
initial_version = initial_metrics["current_version"]

print(f"Initial table version: {initial_version}")
print(f"Initial record count: {initial_metrics['record_count']}")

## 3. Query Table History

In [ ]:
# Query table history
history = delta_table.history().toPandas()
print(f"Table has {len(history)} versions")

# Display the history
print("\nTable history:")
history[['version', 'timestamp', 'operation', 'operationParameters', 'operationMetrics']].head(10)

## 4. Visualize Table History

In [ ]:
# Convert timestamp to datetime
history['timestamp'] = pd.to_datetime(history['timestamp'])

# Create a timeline of operations
plt.figure(figsize=(12, 6))
plt.plot(history['timestamp'], history['version'], marker='o', linestyle='-')
plt.xlabel('Timestamp')
plt.ylabel('Version')
plt.title('Delta Table Version History')
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Create a bar chart of operations
plt.figure(figsize=(10, 6))
operation_counts = history['operation'].value_counts()
operation_counts.plot(kind='bar')
plt.xlabel('Operation')
plt.ylabel('Count')
plt.title('Delta Table Operations')
plt.tight_layout()
plt.show()

## 5. Query Data at a Specific Version

In [ ]:
# Choose a version to query (e.g., the initial version)
version_to_query = 0

# Query data at the specific version
df_at_version = spark.read.format("delta").option("versionAsOf", version_to_query).load(delta_table_path)
count_at_version = df_at_version.count()

print(f"Data at version {version_to_query}:")
print(f"Record count: {count_at_version}")
print("\nSample data:")
df_at_version.show(5)

# Check schema at this version
print("\nSchema at this version:")
df_at_version.printSchema()

## 6. Query Data at a Specific Timestamp

In [ ]:
# Choose a timestamp to query (e.g., 1 hour ago)
timestamp_to_query = (datetime.now() - timedelta(hours=1)).strftime("%Y-%m-%d %H:%M:%S")

# Query data at the specific timestamp
df_at_timestamp = spark.read.format("delta").option("timestampAsOf", timestamp_to_query).load(delta_table_path)
count_at_timestamp = df_at_timestamp.count()

print(f"Data at timestamp {timestamp_to_query}:")
print(f"Record count: {count_at_timestamp}")
print("\nSample data:")
df_at_timestamp.show(5)

# Find the version that corresponds to this timestamp
version_at_timestamp = history[history['timestamp'] <= pd.Timestamp(timestamp_to_query)]['version'].max()
print(f"\nThis timestamp corresponds to version {version_at_timestamp}")

## 7. Intentionally Write Bad Data

In [ ]:
# Generate corrupted data
print("Generating corrupted data...")
corrupted_data = utils.generate_test_data(num_records=200, scenario='corrupted')
corrupted_df = spark.createDataFrame(corrupted_data)

# Add a marker to identify the corrupted data
corrupted_df = corrupted_df.withColumn("Data_Quality", lit("corrupted"))

# Write the corrupted data to the Delta table
print("Writing corrupted data to the Delta table...")
corrupted_df.write.format("delta").mode("append").save(delta_table_path)

# Get metrics after writing corrupted data
corrupted_metrics = utils.log_delta_table_metrics(spark, delta_table_path)
corrupted_version = corrupted_metrics["current_version"]

print(f"\nTable version after writing corrupted data: {corrupted_version}")
print(f"Record count after writing corrupted data: {corrupted_metrics['record_count']}")
print(f"Records added: {corrupted_metrics['record_count'] - initial_metrics['record_count']}")

# Verify the corrupted data was written
print("\nVerifying corrupted data:")
spark.read.format("delta").load(delta_table_path) \
    .filter(col("Data_Quality") == "corrupted") \
    .select("Order ID", "Sales", "Quantity", "Data_Quality") \
    .show(5)

## 8. Detect Data Quality Issues

In [ ]:
# Check for negative sales and quantities
print("Checking for data quality issues...")
current_df = spark.read.format("delta").load(delta_table_path)

# Count records with negative sales
negative_sales_count = current_df.filter(col("Sales") < 0).count()
print(f"Records with negative sales: {negative_sales_count}")

# Count records with negative quantities
negative_quantity_count = current_df.filter(col("Quantity") < 0).count()
print(f"Records with negative quantities: {negative_quantity_count}")

# Show some of the problematic records
print("\nSample of problematic records:")
current_df.filter((col("Sales") < 0) | (col("Quantity") < 0)) \
    .select("Order ID", "Sales", "Quantity", "Data_Quality") \
    .show(5)

## 9. Compare Data Across Versions

In [ ]:
# Compare data quality across versions
print("Comparing data quality across versions...")

# Function to check data quality for a specific version
def check_data_quality(version):
    df = spark.read.format("delta").option("versionAsOf", version).load(delta_table_path)
    total_count = df.count()
    negative_sales = df.filter(col("Sales") < 0).count()
    negative_quantity = df.filter(col("Quantity") < 0).count()
    corrupted_count = df.filter(col("Data_Quality") == "corrupted").count() if "Data_Quality" in df.columns else 0
    
    return {
        "version": version,
        "total_count": total_count,
        "negative_sales": negative_sales,
        "negative_quantity": negative_quantity,
        "corrupted_count": corrupted_count,
        "negative_sales_pct": (negative_sales / total_count) * 100 if total_count > 0 else 0,
        "negative_quantity_pct": (negative_quantity / total_count) * 100 if total_count > 0 else 0
    }

# Check data quality for multiple versions
versions_to_check = [0, initial_version, corrupted_version]
quality_results = [check_data_quality(v) for v in versions_to_check]

# Display results
quality_df = pd.DataFrame(quality_results)
print("Data quality comparison across versions:")
quality_df

## 10. Rollback to a Previous Version

In [ ]:
# Rollback to the version before corrupted data was written
version_to_restore = initial_version
print(f"Rolling back to version {version_to_restore}...")

# Perform the rollback
spark.sql(f"RESTORE TABLE delta.`{delta_table_path}` VERSION AS OF {version_to_restore}")

# Get metrics after rollback
restored_metrics = utils.log_delta_table_metrics(spark, delta_table_path)
restored_version = restored_metrics["current_version"]

print(f"\nTable version after rollback: {restored_version}")
print(f"Record count after rollback: {restored_metrics['record_count']}")

# Verify the corrupted data is gone
print("\nVerifying corrupted data is gone:")
current_df = spark.read.format("delta").load(delta_table_path)
negative_sales_count = current_df.filter(col("Sales") < 0).count()
negative_quantity_count = current_df.filter(col("Quantity") < 0).count()

print(f"Records with negative sales: {negative_sales_count}")
print(f"Records with negative quantities: {negative_quantity_count}")

# Check if the Data_Quality column still exists
has_data_quality = "Data_Quality" in current_df.columns
corrupted_count = current_df.filter(col("Data_Quality") == "corrupted").count() if has_data_quality else 0
print(f"Records marked as corrupted: {corrupted_count}")

## 11. Query Table History After Rollback

In [ ]:
# Query table history after rollback
history_after_rollback = delta_table.history().toPandas()
print(f"Table has {len(history_after_rollback)} versions after rollback")

# Display the history
print("\nTable history after rollback:")
history_after_rollback[['version', 'timestamp', 'operation', 'operationParameters']].head(10)

## 12. Time Travel for Analytics

In [ ]:
# Demonstrate time travel for analytics
print("Demonstrating time travel for analytics...")

# Get all versions
all_versions = history_after_rollback['version'].tolist()

# Function to get sales metrics for a specific version
def get_sales_metrics(version):
    df = spark.read.format("delta").option("versionAsOf", version).load(delta_table_path)
    
    # Calculate sales metrics
    metrics = df.agg(
        expr("sum(Sales)").alias("total_sales"),
        expr("avg(Sales)").alias("avg_sales"),
        expr("sum(Profit)").alias("total_profit"),
        expr("count(*)").alias("record_count")
    ).collect()[0]
    
    return {
        "version": version,
        "total_sales": metrics["total_sales"],
        "avg_sales": metrics["avg_sales"],
        "total_profit": metrics["total_profit"],
        "record_count": metrics["record_count"]
    }

# Get sales metrics for multiple versions
# Use a subset of versions to avoid too many queries
versions_to_analyze = sorted(all_versions)[::max(1, len(all_versions) // 5)][:5]
sales_metrics = [get_sales_metrics(v) for v in versions_to_analyze]

# Display results
sales_df = pd.DataFrame(sales_metrics)
print("Sales metrics across versions:")
sales_df

# Visualize the metrics
plt.figure(figsize=(12, 6))
plt.plot(sales_df['version'], sales_df['total_sales'], marker='o', label='Total Sales')
plt.plot(sales_df['version'], sales_df['total_profit'], marker='s', label='Total Profit')
plt.xlabel('Version')
plt.ylabel('Amount')
plt.title('Sales and Profit Across Versions')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 13. Optimize Storage While Preserving Time Travel

In [ ]:
# Check current retention period
retention_period = spark.sql(f"SHOW TBLPROPERTIES delta.`{delta_table_path}` ('delta.logRetentionDuration')").collect()
print(f"Current log retention period: {retention_period[0][1]}")

# Set a reasonable retention period (e.g., 30 days)
print("Setting log retention period to 30 days...")
spark.sql(f"ALTER TABLE delta.`{delta_table_path}` SET TBLPROPERTIES ('delta.logRetentionDuration' = '30 days')")

# Check file sizes before vacuum
before_vacuum_metrics = utils.monitor_file_metrics(spark, delta_table_path, "before vacuum")

# Run vacuum with dry run to see what would be deleted
print("\nRunning VACUUM DRY RUN to see what would be deleted...")
# Note: We're using a short retention period for demonstration purposes
# In production, never set this below 7 days, especially with streaming workloads
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
vacuum_dry_run = spark.sql(f"VACUUM delta.`{delta_table_path}` RETAIN 0 HOURS DRY RUN").collect()
num_files_to_delete = len(vacuum_dry_run)
print(f"Number of files that would be deleted: {num_files_to_delete}")

# Reset retention check for safety
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "true")

print("\nBest practices for optimizing storage while preserving time travel:")
print("1. Set an appropriate retention period based on your time travel needs")
print("2. Run OPTIMIZE regularly to compact small files")
print("3. Run VACUUM with a safe retention period (at least 7 days)")
print("4. Monitor storage usage and adjust retention period as needed")
print("5. Consider using Delta Lake's data skipping capabilities for efficient time travel queries")

## 14. Summary

In this notebook, we've demonstrated Delta Lake's time travel capabilities:

1. Querying table history to understand changes over time
2. Accessing data at specific versions or timestamps
3. Detecting and analyzing data quality issues
4. Rolling back to a previous version to recover from bad data
5. Using time travel for analytical purposes
6. Optimizing storage while preserving time travel capabilities

These capabilities make Delta Lake a powerful tool for data governance, auditing, and recovery scenarios. Time travel allows you to understand how your data has changed over time, recover from errors, and perform point-in-time analysis.